# Fine-tune BioBERT on BC5CDR + Food Data

This notebook is designed to run **both locally (using your local GPU RTX 3080 or CPU)** and on **Google Colab**.
- If run locally, it will automatically download the BC5CDR and FoodBase datasets from Hugging Face, parse them into BIO format, and save them in `data/en/` in your workspace.
- If run on Colab, it will mount Google Drive and look/save to `/content/drive/MyDrive/nutrition-rag`.
- It dynamically configures GPU/CPU settings, batch sizes, and FP16 half-precision based on your hardware.

In [1]:
# Install dependencies if missing (e.g. on Colab or fresh environment)
!pip install -q evaluate seqeval datasets transformers accelerate

## Setup — Environment & Data Paths

In [2]:
import os
import json
import urllib.request
from datasets import load_dataset

# 1. Determine environment and base paths
try:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_BASE = "/content/drive/MyDrive/nutrition-rag"
    is_colab = True
except ImportError:
    is_colab = False
    # Local environment - find data and model directories relative to notebooks folder
    print("Running locally.")
    # Target local data folder: HEALTHCARE-RAG/data/en
    DRIVE_BASE = os.path.abspath(os.path.join(os.getcwd(), "../../data/en"))
    # Fallback checks
    if not os.path.exists(DRIVE_BASE):
        DRIVE_BASE = os.path.abspath(os.path.join(os.getcwd(), "data/en"))
    if not os.path.exists(DRIVE_BASE):
        DRIVE_BASE = os.path.abspath(".")

BC5CDR_PATH = os.path.join(DRIVE_BASE, "bc5cdr_bio.jsonl")
FOOD_PATH = os.path.join(DRIVE_BASE, "food_bio.jsonl")

# Target output model folder: HEALTHCARE-RAG/models/ner_bert
if is_colab:
    OUTPUT_DIR = "/content/drive/MyDrive/nutrition-rag/ner_bert"
else:
    OUTPUT_DIR = os.path.abspath(os.path.join(os.getcwd(), "../../models/ner_bert"))
    if not os.path.exists(os.path.dirname(OUTPUT_DIR)):
        OUTPUT_DIR = os.path.abspath("models/ner_bert")

os.makedirs(DRIVE_BASE, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 2. Automatically prepare BC5CDR dataset if missing
def prepare_bc5cdr():
    if os.path.exists(BC5CDR_PATH):
        print(f"-> BC5CDR file already exists: {BC5CDR_PATH}")
        return
    
    print("-> BC5CDR file not found locally. Downloading from Hugging Face hub...")
    # Load BC5CDR JSON files from Hugging Face tner repository (which bypasses the script issue)
    ds = load_dataset('json', data_files={
        'train': 'https://huggingface.co/datasets/tner/bc5cdr/raw/main/dataset/train.json',
        'validation': 'https://huggingface.co/datasets/tner/bc5cdr/raw/main/dataset/valid.json',
        'test': 'https://huggingface.co/datasets/tner/bc5cdr/raw/main/dataset/test.json'
    })
    
    # Map the tags to labels
    # tner/bc5cdr tags: {'O': 0, 'B-Chemical': 1, 'B-Disease': 2, 'I-Disease': 3, 'I-Chemical': 4}
    # Map B-Chemical -> B-NUTRIENT, I-Chemical -> I-NUTRIENT, B-Disease -> B-DISEASE, I-Disease -> I-DISEASE
    label_map = {
        0: 'O',
        1: 'B-NUTRIENT',
        2: 'B-DISEASE',
        3: 'I-DISEASE',
        4: 'I-NUTRIENT'
    }
    
    written_count = 0
    with open(BC5CDR_PATH, 'w', encoding='utf-8') as f:
        for split in ['train', 'validation', 'test']:
            for item in ds[split]:
                mapped_labels = [label_map.get(t, 'O') for t in item['tags']]
                row = {
                    'tokens': item['tokens'],
                    'labels': mapped_labels
                }
                f.write(json.dumps(row) + '\n')
                written_count += 1
    print(f"-> Saved {written_count} parsed sentences to {BC5CDR_PATH}")

# 3. Automatically prepare FoodBase dataset if missing
def prepare_food_data():
    if os.path.exists(FOOD_PATH):
        print(f"-> Food data file already exists: {FOOD_PATH}")
        return
        
    print("-> Food data file not found locally. Downloading FoodBase from Hugging Face...")
    ds = load_dataset("Dizex/FoodBase")
    
    written_count = 0
    with open(FOOD_PATH, 'w', encoding='utf-8') as f:
        for split in ['train', 'val']:
            for item in ds[split]:
                row = {
                    'tokens': item['nltk_tokens'],
                    'labels': item['iob_tags']
                }
                f.write(json.dumps(row) + '\n')
                written_count += 1
    print(f"-> Saved {written_count} parsed sentences to {FOOD_PATH}")

prepare_bc5cdr()
prepare_food_data()

print("\n--- Setup Complete ---")
print(f"BC5CDR path: {BC5CDR_PATH}")
print(f"Food path:   {FOOD_PATH}")
print(f"Output path: {OUTPUT_DIR}")

d:\Data\File for Google Drive real\Project\Nutrition_RAG\HEALTHCARE-RAG\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Running locally.
-> BC5CDR file already exists: d:\Data\File for Google Drive real\Project\Nutrition_RAG\HEALTHCARE-RAG\data\en\bc5cdr_bio.jsonl
-> Food data file already exists: d:\Data\File for Google Drive real\Project\Nutrition_RAG\HEALTHCARE-RAG\data\en\food_bio.jsonl

--- Setup Complete ---
BC5CDR path: d:\Data\File for Google Drive real\Project\Nutrition_RAG\HEALTHCARE-RAG\data\en\bc5cdr_bio.jsonl
Food path:   d:\Data\File for Google Drive real\Project\Nutrition_RAG\HEALTHCARE-RAG\data\en\food_bio.jsonl
Output path: d:\Data\File for Google Drive real\Project\Nutrition_RAG\HEALTHCARE-RAG\models\ner_bert


## Setup Labels

In [3]:
import json
import warnings
import numpy as np
import evaluate
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    TrainingArguments,
    Trainer,
)

warnings.filterwarnings("ignore")

MODEL_CHECKPOINT = "dmis-lab/biobert-base-cased-v1.2"
LABEL_LIST = ["O", "B-NUTRIENT", "I-NUTRIENT", "B-DISEASE", "I-DISEASE", "B-FOOD", "I-FOOD"]
label2id = {l: i for i, l in enumerate(LABEL_LIST)}
id2label  = {i: l for l, i in label2id.items()}

print(f"Labels: {label2id}")

Labels: {'O': 0, 'B-NUTRIENT': 1, 'I-NUTRIENT': 2, 'B-DISEASE': 3, 'I-DISEASE': 4, 'B-FOOD': 5, 'I-FOOD': 6}


## Load & Merge Data

In [4]:
def load_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

bc5cdr_rows = load_jsonl(BC5CDR_PATH)
food_rows = load_jsonl(FOOD_PATH)

# --- Data Augmentation for Conversational QA contexts ---
# Extract unique food entities from recipe food_rows
unique_foods = set()
for r in food_rows:
    tokens = r["tokens"]
    labels = r["labels"]
    current_food = []
    for t, l in zip(tokens, labels):
        if l == "B-FOOD":
            if current_food:
                unique_foods.add(tuple(current_food))
            current_food = [t]
        elif l == "I-FOOD":
            if current_food:
                current_food.append(t)
        else:
            if current_food:
                unique_foods.add(tuple(current_food))
                current_food = []
    if current_food:
        unique_foods.add(tuple(current_food))

print(f"Extracted {len(unique_foods)} unique food entities from recipes.")

# Generate synthetic conversational QA templates
synthetic_rows = []
for food_tuple in unique_foods:
    food_tokens = list(food_tuple)
    food_len = len(food_tokens)
    food_labels = ["B-FOOD"] + ["I-FOOD"] * (food_len - 1)
    
    # T1: What is the nutrition values of [FOOD]?
    synthetic_rows.append({
        "tokens": ["What", "is", "the", "nutrition", "values", "of"] + food_tokens + ["?"],
        "labels": ["O", "O", "O", "O", "O", "O"] + food_labels + ["O"]
    })
    # T2: Show me the protein in [FOOD]
    synthetic_rows.append({
        "tokens": ["Show", "me", "the", "protein", "in"] + food_tokens,
        "labels": ["O", "O", "O", "B-NUTRIENT", "O"] + food_labels
    })
    # T3: How many calories are in [FOOD]?
    synthetic_rows.append({
        "tokens": ["How", "many", "calories", "are", "in"] + food_tokens + ["?"],
        "labels": ["O", "O", "B-NUTRIENT", "O", "O"] + food_labels + ["O"]
    })
    # T4: what about [FOOD]?
    synthetic_rows.append({
        "tokens": ["what", "about"] + food_tokens + ["?"],
        "labels": ["O", "O"] + food_labels + ["O"]
    })
    # T5: Is [FOOD] good for diabetes?
    synthetic_rows.append({
        "tokens": ["Is"] + food_tokens + ["good", "for", "diabetes", "?"],
        "labels": ["O"] + food_labels + ["O", "O", "B-DISEASE", "O"]
    })
    # T6: Tell me about [FOOD]
    synthetic_rows.append({
        "tokens": ["Tell", "me", "about"] + food_tokens,
        "labels": ["O", "O", "O"] + food_labels
    })

print(f"Generated {len(synthetic_rows)} synthetic QA rows.")

# --- Add General Domain Negative Samples to teach the model 'O' (no-entity) class ---
negative_samples = []
subjects = [
    "the capital of France", "the speed of light", "the height of Mount Everest", "the population of Tokyo",
    "the author of Hamlet", "the president of America", "the CEO of Microsoft", "the size of the solar system",
    "the rules of soccer", "the definition of inflation", "the behavior of black holes", "the concept of quantum physics",
    "the development of the internet", "the architecture of the Eiffel Tower", "the structure of the bridge",
    "the meaning of life", "the history of music", "the play Romeo and Juliet", "the painting Mona Lisa",
    "the discovery of America", "the launch of the rocket", "the software development lifecycle",
    "the database schema design", "the git commit message format", "the web browser search history",
    "the weather forecast for tomorrow", "the flight schedule to London", "the train ticket booking system",
    "the currency exchange rate", "the stock market index value", "the rules of chess", "the capital of Japan",
    "the history of the Roman Empire", "the distance to Mars", "the length of the Amazon river",
    "the author of Harry Potter", "the inventor of the light bulb", "the prime minister of Canada",
    "the founder of Apple", "the design of the smartphone", "the features of the new car",
    "the schedule of the conference", "the results of the match", "the location of the museum",
    "the price of the gold", "the oil production rate", "the interest rate of the bank",
    "the details of the project", "the architecture of the building", "the layout of the keyboard",
    "the language of Brazil", "the flag of Germany", "the population of India", "the history of the movie",
    "the song of the artist", "the performance of the computer", "the speed of the network",
    "the configuration of the server", "the documentation of the API", "the source code of the app"
]
templates = [
    "What is {subject}?",
    "How do you explain {subject}?",
    "Who is responsible for {subject}?",
    "Can you explain {subject}?",
    "Show me {subject}.",
    "Tell me about {subject}.",
    "What do you know about {subject}?",
    "Why is {subject} important?",
    "Where can I find {subject}?",
    "Is {subject} available online?",
    "Please describe {subject}.",
    "Who discovered {subject}?",
    "What is the history of {subject}?",
    "How does {subject} work?",
    "Can we look at {subject}?"
]
for sub in subjects:
    for temp in templates:
        text = temp.format(subject=sub)
        tokens = text.strip().split()
        if len(tokens) >= 3:
            negative_samples.append({
                "tokens": tokens,
                "labels": ["O"] * len(tokens)
            })
print(f"-> Generated {len(negative_samples)} clean negative samples synthetically (zero-overlap vocabulary).")

all_rows = bc5cdr_rows + food_rows + synthetic_rows + negative_samples

import random
random.seed(42)
random.shuffle(all_rows)

# 70/30 split
split_idx = int(len(all_rows) * 0.7)
train_rows, val_rows = all_rows[:split_idx], all_rows[split_idx:]

def rows_to_dataset(rows):
    return Dataset.from_dict({
        "tokens": [r["tokens"] for r in rows],
        "labels": [[label2id[l] for l in r["labels"]] for r in rows],
    })

train_ds = rows_to_dataset(train_rows)
val_ds = rows_to_dataset(val_rows)
print(f"Combined Train (with Augmentation & Negatives): {len(train_ds)} | Val: {len(val_ds)}")

Extracted 1860 unique food entities from recipes.
Generated 11160 synthetic QA rows.
-> Generated 900 clean negative samples synthetically (zero-overlap vocabulary).
Combined Train (with Augmentation & Negatives): 20498 | Val: 8785


## Preprocessing — Subword Alignment

In [5]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT)

def tokenize_and_align(batch):
    encoding = tokenizer(batch["tokens"], is_split_into_words=True, truncation=True, max_length=512, padding=False)
    aligned_labels = []
    for i, label_ids in enumerate(batch["labels"]):
        word_ids = encoding.word_ids(batch_index=i)
        prev_word_id = None
        row_labels = []
        for word_id in word_ids:
            if word_id is None:
                row_labels.append(0)
            elif word_id != prev_word_id:
                row_labels.append(label_ids[word_id])
            else:
                parent_label = id2label[label_ids[word_id]]
                if parent_label.startswith("B-"):
                    child_label = "I-" + parent_label[2:]
                    row_labels.append(label2id[child_label])
                else:
                    row_labels.append(label_ids[word_id])
            prev_word_id = word_id
        aligned_labels.append(row_labels)
    encoding["labels"] = aligned_labels
    return encoding

train_tok = train_ds.map(tokenize_and_align, batched=True, remove_columns=["tokens", "labels"])
val_tok = val_ds.map(tokenize_and_align, batched=True, remove_columns=["tokens", "labels"])

Map: 100%|██████████| 8785/8785 [00:00<00:00, 11846.07 examples/s]


## Training

In [6]:
import os, sys
if not os.path.exists('configs/config.yaml'):
    os.chdir('../..')
sys.path.insert(0, '.')

from src.en.ner import BertCRFForTokenClassification
import torch
import evaluate

# Dynamically select GPU if available
use_cuda = torch.cuda.is_available()
print(f"CUDA Available: {use_cuda}")
if use_cuda:
    print(f"Using GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Using CPU. Training might be very slow.")

seqeval = evaluate.load("seqeval")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Run Viterbi decoding via the global model's CRF module
    with torch.no_grad():
        emissions_t = torch.tensor(logits)
        mask_t = torch.tensor(labels != -100)
        decoded_paths = model.crf.decode(emissions_t, mask_t)
    
    true_labels, true_preds = [], []
    for path, label_row in zip(decoded_paths, labels):
        row_true = [id2label[l] for l in label_row if l != -100]
        row_pred = [id2label[p] for p in path]
        min_l = min(len(row_true), len(row_pred))
        true_labels.append(row_true[:min_l])
        true_preds.append(row_pred[:min_l])
    
    result = seqeval.compute(predictions=true_preds, references=true_labels)
    return {
        "f1": result["overall_f1"],
        "precision": result["overall_precision"],
        "recall": result["overall_recall"]
    }

# Load custom BertCRF model instead of default token classification model
model = BertCRFForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABEL_LIST),
    id2label=id2label,
    label2id=label2id,
    ignore_mismatched_sizes=True
)
# CRITICAL: re-initialize classifier + CRF head after from_pretrained.
# from_pretrained corrupts MISSING params (CRF transitions).
model.reinit_head()
data_collator = DataCollatorForTokenClassification(tokenizer)

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=3,
    per_device_train_batch_size=16 if use_cuda else 4,
    per_device_eval_batch_size=32 if use_cuda else 8,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    max_grad_norm=1.0,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=False,  # CRF loss is numerically unstable in FP16
    use_cpu=not use_cuda,
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tok,
    eval_dataset=val_tok,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics
)
trainer.train()

CUDA Available: True
Using GPU: NVIDIA GeForce RTX 3080


[transformers] You passed `num_labels=7` which is incompatible to the `id2label` map of length `2`.
Loading weights: 100%|██████████| 197/197 [00:00<00:00, 27398.30it/s]
[transformers] BertCRFForTokenClassification LOAD REPORT from: dmis-lab/biobert-base-cased-v1.2
Key                                        | Status     | 
-------------------------------------------+------------+-
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weigh

Epoch,Training Loss,Validation Loss,F1,Precision,Recall
1,2.037492,1.472164,0.929641,0.920437,0.939030
2,1.074549,1.561840,0.947576,0.939312,0.955987
3,0.653402,1.722189,0.953537,0.945531,0.961680


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.12it/s]


TrainOutput(global_step=3846, training_loss=1.832639482947482, metrics={'train_runtime': 3033.1444, 'train_samples_per_second': 20.274, 'train_steps_per_second': 1.268, 'total_flos': 3006355266623856.0, 'train_loss': 1.832639482947482, 'epoch': 3.0})

## Evaluation and Saving

In [7]:
metrics = trainer.evaluate()
print(f"\nValidation results: F1: {metrics['eval_f1']:.4f}")
trainer.save_model(OUTPUT_DIR)
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"Saved to {OUTPUT_DIR}")

Training Loss,Validation Loss,Epoch,F1,Precision,Recall
0.653402,1.722189,3,0.953537,0.945531,0.961680



Validation results: F1: 0.9535


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  1.28it/s]

Saved to d:\Data\File for Google Drive real\Project\Nutrition_RAG\HEALTHCARE-RAG\models\ner_bert
